# Technique 2

## Case 1: Earth to Asteroid Dionysus

Ten shooting strategies on the same 5000 random initial guesses: single shooting at $\epsilon=1$ and at the target level $\epsilon_f=10^{-5}$, stepwise homotopy $\epsilon=1,10^{-1},\ldots,10^{-5}$ with backtracking, the ungated augmented formulation, and augmented shooting with the exponential and rational gates for $p=1$, $2$ and $4$. Convergence is judged by the physical shooting residual at the target smoothing level.

In [1]:
import os
import sys
import platform

for name in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS',
             'VECLIB_MAXIMUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[name] = '1'

import numpy as np
import scipy
import heyoka as hy
from IPython.display import display, Markdown

hy.set_nthreads(1)

In [2]:
# =========================
# Timing Settings
# =========================
from collections import Counter

cpu_samples = []
fast_cpus = []
if sys.platform == 'darwin':
    import ctypes
    import plistlib
    import subprocess
    tree = plistlib.loads(subprocess.check_output(
        ['/usr/sbin/ioreg', '-a', '-l', '-p', 'IODeviceTree', '-n', 'cpus', '-r']))
    pending = list(tree)
    while pending:
        node = pending.pop()
        if node.get('cluster-type') == b'P\x00' and 'cpu-id' in node:
            fast_cpus.append(int.from_bytes(node['cpu-id'], 'little'))
        pending.extend(node.get('IORegistryEntryChildren', []))
    lib = ctypes.CDLL('/usr/lib/libSystem.B.dylib')
    lib.pthread_set_qos_class_self_np.argtypes = [ctypes.c_uint, ctypes.c_int]
    lib.pthread_cpu_number_np.argtypes = [ctypes.POINTER(ctypes.c_size_t)]

def set_timing_priority():
    if sys.platform == 'darwin':
        assert lib.pthread_set_qos_class_self_np(0x21, 0) == 0

def sample_cpu():
    if sys.platform == 'darwin':
        cpu = ctypes.c_size_t()
        assert lib.pthread_cpu_number_np(ctypes.byref(cpu)) == 0
        cpu_samples.append(cpu.value)

environment = dict(python=sys.version, executable=sys.executable,
                   platform=platform.platform(), machine=platform.machine(),
                   numpy=np.__version__, scipy=scipy.__version__, heyoka=hy.__version__,
                   threads=1, fast_cpus=sorted(fast_cpus), strict_cpu_pinning=False)
print(environment)

records_by_method = {}
results_by_method = {}

{'python': '3.12.14 (main, Sep  3 2026, 03:46:19) [Clang 21.1.8 ]', 'executable': '/Users/zhong/Documents/Codex/miniforge3/envs/gtoc13/bin/python', 'platform': 'macOS-26.5.1-arm64-arm-64bit', 'machine': 'arm64', 'numpy': '2.5.3', 'scipy': '1.18.0', 'heyoka': '7.10.1', 'threads': 1, 'fast_cpus': [12, 13, 14, 15, 16, 17], 'strict_cpu_pinning': False}


In [3]:
import numpy as np

# =========================
# Setup (SI Units)
# =========================
AU2m = 149597870700 # JPL 
day2s = 86400.0     # JPL

tof = 3534.0*day2s # [s]

m0 = 4000.0 # [kg]
Tmax = 0.32 # [N]
Isp = 3000  # [s]

mu = 132712440018*1e9   # [m3∕s2]
g0 = 9.8065             # [m∕s2]

# Initial and terminal MEE (p, f, g, h, k, L); terminal L includes five revolutions.
mee0 = [149554268256.46957, -0.003766786954568226, 0.016286832739416963, -7.702048962437313e-06, 6.183156622292267e-07, 1.5955219194574601]

meef = [232433083985.70743, 0.1530290696088389, -0.5199481742007113, 0.01618310223871937, 0.11813952745106715, 33.763869726200596]

# =========================
# Output dimensional MEE in scientific notation
# =========================
print("mee0 (dimensional) =", np.array2string(
    np.asarray(mee0),
    formatter={"float_kind": lambda x: f"{x:.8e}"}
))
print("meef (dimensional) =", np.array2string(
    np.asarray(meef),
    formatter={"float_kind": lambda x: f"{x:.8e}"}
))

# =========================
# Nondimensionalize
# =========================
# Nondimension units
LUnit = AU2m
TUnit = np.sqrt(LUnit * LUnit * LUnit / mu)
MUnit = m0

VUnit = LUnit / TUnit # VUnit to m/s
AUnit = VUnit / TUnit # AUnit to m/s^2

# x0_nu, xf_nu, t0_nu, tf_nu, m0_nu, mu_nu, c1=Tmax_nu, c2=Isp_nu*g0_nu 
x0_nu = mee0
xf_nu = meef
t0_nu = 0.0
tf_nu = t0_nu+tof
m0_nu = m0
mu_nu = mu
Tmax_nu = Tmax
Isp_nu = Isp
g0_nu = g0

x0_nu[0] = x0_nu[0] / LUnit
xf_nu[0] = xf_nu[0] / LUnit

t0_nu = t0_nu / TUnit
tf_nu = tf_nu / TUnit
m0_nu = m0_nu / MUnit
mu_nu = mu_nu * TUnit * TUnit / LUnit / LUnit / LUnit
Tmax_nu = Tmax_nu / MUnit / AUnit
Isp_nu = Isp_nu / TUnit
g0_nu = g0_nu / AUnit

c1 = Tmax_nu
c2 = Isp_nu*g0_nu


mee0 (dimensional) = [1.49554268e+11 -3.76678695e-03 1.62868327e-02 -7.70204896e-06
 6.18315662e-07 1.59552192e+00]
meef (dimensional) = [2.32433084e+11 1.53029070e-01 -5.19948174e-01 1.61831022e-02
 1.18139527e-01 3.37638697e+01]


In [4]:
import twobody_MEE_hamilton_log

ta = twobody_MEE_hamilton_log.ta

In [5]:
# =========================
# Numerical safeguards
# =========================
# A propagation that does not reach the terminal time within integration_max_steps steps, a nonfinite
# state, or a homotopy variable below the admissible range returns a large finite residual, so that
# MINPACK rejects the trial point and shrinks its trust region instead of evaluating pathological
# trajectories. Converged solutions use a few hundred steps.
integration_max_steps = 5000
BAD_RESIDUAL = 1.0e20
Q_MIN_ALLOWED = -9.0          # eps(q) above 10 eps0 is meaningless; the lower side is clipped at eps_f in the dynamics

def _bad_residual(n):
    # Residual penalty for an invalid trial
    return np.full(n, BAD_RESIDUAL, dtype=float)

# =========================
# Define the shooting function
# =========================
def shooting(lam, eps = 1.0e-5, lambda0 = 1.0, out = False, Time_out = None):
    # lam[0:7] -> initial co-states [lp,lf,lg,lh,lk,lL,lm]
    # function uses global variables ta, t0_nu, x0_nu, m0_nu, tf_nu, xf_nu,
    # mu_nu, c1, c2
    ta.pars[:] = [mu_nu, c1, c2, eps, lambda0]
    ta.time = t0_nu
    ta.state[:7] = x0_nu + [m0_nu]
    ta.state[7:14] = lam[:]

    result = ta.propagate_until(tf_nu, max_steps=integration_max_steps)
    if ta.time != tf_nu or not np.all(np.isfinite(ta.state)):
        return _bad_residual(7)

    # Construct 7 equations: 6 terminal mee errors + 1 terminal co-state lm = 0
    retval = [
        ta.state[0] - xf_nu[0],
        ta.state[1] - xf_nu[1],
        ta.state[2] - xf_nu[2],
        ta.state[3] - xf_nu[3],
        ta.state[4] - xf_nu[4],
        ta.state[5] - xf_nu[5],
        ta.state[13]
    ]

    if out:
        print("  Final Mass:", ta.state[6]*MUnit)

    if Time_out is not None:
        test = Time_out

    return retval

def shooting_augmented(lam_augmented, eps0 = 1.0, epsf = 1.0e-5, lambda0 = 1.0, out = False, Time_out = None):
    # Unknowns: seven initial co-states and the homotopy variable q
    # function uses global variables ta, t0_nu, x0_nu, m0_nu, tf_nu, xf_nu,
    # mu_nu, c1, c2

    # lam_augmented[0:7] -> initial co-states [lp,lf,lg,lh,lk,lL,lm]
    # lam_augmented[7]   -> homotopy variable q = (eps0 - eps) / (eps0 - epsf), i.e. eps is LINEAR in q:
    #   q = 0 -> eps = eps0
    #   q = 1 -> eps = epsf

    q = lam_augmented[7]
    if not np.isfinite(q) or q < Q_MIN_ALLOWED:
        return _bad_residual(8)

    # The dynamics are integrated with eps never sharper than the target; the homotopy residual uses the raw q
    eps = eps0 + min(q, 1.0) * (epsf - eps0)

    ta.pars[:] = [mu_nu, c1, c2, eps, lambda0]
    ta.time = t0_nu
    ta.state[:7] = x0_nu + [m0_nu]
    ta.state[7:14] = lam_augmented[:-1]

    result = ta.propagate_until(tf_nu, max_steps=integration_max_steps)
    if ta.time != tf_nu or not np.all(np.isfinite(ta.state)):
        return _bad_residual(8)

    # Construct 8 equations:
    # 6 terminal MEE errors + lm(tf) = 0 + q = 1
    retval = [
        ta.state[0] - xf_nu[0],
        ta.state[1] - xf_nu[1],
        ta.state[2] - xf_nu[2],
        ta.state[3] - xf_nu[3],
        ta.state[4] - xf_nu[4],
        ta.state[5] - xf_nu[5],
        ta.state[13],
        q - 1.0,              # drive eps to epsf independently of the shooting residual
    ]

    if out:
        print("  Final Mass:", ta.state[6]*MUnit)
        print(f"  Final eps : {eps0 + q*(epsf - eps0):.16e}  (integrated with eps = {eps:.16e}, q = {q:.8f})")

    if Time_out is not None:
        test = Time_out

    return retval

def shooting_augmented_gate(
    lam_augmented,
    eps0 = 1.0,             # initial smoothing level
    epsf = 1.0e-5,          # target smoothing level
    rho0 = 1.0e-2,          # gate threshold
    delta_rho = 1.0e-8,     # regularization for rho
    gate_p = 2.0,           # gate exponent
    gate_kind = "gaussian", # gate option
    lambda0 = 1.0,
    out = False,
    Time_out = None
):

    # lam_augmented[0:7] -> initial co-states [lp,lf,lg,lh,lk,lL,lm]
    # lam_augmented[7]   -> homotopy variable q = (eps0 - eps) / (eps0 - epsf), i.e. eps is LINEAR in q:
    #   q = 0 -> eps = eps0
    #   q = 1 -> eps = epsf

    q = lam_augmented[7]
    if not np.isfinite(q) or q < Q_MIN_ALLOWED:
        return _bad_residual(8)

    # The dynamics are integrated with eps never sharper than the target; the gate equation uses the raw q
    eps = eps0 + min(q, 1.0) * (epsf - eps0)

    ta.pars[:] = [mu_nu, c1, c2, eps, lambda0]
    ta.time = t0_nu
    ta.state[:7] = x0_nu + [m0_nu]
    ta.state[7:14] = lam_augmented[:-1]

    result = ta.propagate_until(tf_nu, max_steps=integration_max_steps)
    if ta.time != tf_nu or not np.all(np.isfinite(ta.state)):
        return _bad_residual(8)

    # Terminal residual vector Phi
    Phi = np.array([
        ta.state[0] - xf_nu[0],
        ta.state[1] - xf_nu[1],
        ta.state[2] - xf_nu[2],
        ta.state[3] - xf_nu[3],
        ta.state[4] - xf_nu[4],
        ta.state[5] - xf_nu[5],
        ta.state[13]
    ], dtype=float)

    # Regularized RMS residual
    rho = np.sqrt(delta_rho**2 + np.mean(Phi * Phi))

    # Exponential and rational gate functions
    # gate switch
    if gate_kind == "gaussian":
        w = np.exp(- (rho / rho0) ** gate_p)
    elif gate_kind == "rational":
        w = 1.0 / (1.0 + (rho / rho0) ** gate_p)
    else:
        raise ValueError(f"Unknown gate_kind: {gate_kind}")

    # augmented residual, q measured as the progress in eps, (eps0 - eps)/(eps0 - epsf):
    # when rho is large -> w ~ 0 -> q ~ 0 -> eps ~ eps0
    # when rho is small -> w ~ 1 -> q ~ 1 -> eps ~ epsf
    retval = list(Phi) + [q - w]

    if out:
        # Output the smoothing parameter and residual norm
        print(f"  Final Mass: {ta.state[6]*MUnit}")
        print(f"  Final eps : {eps0 + q*(epsf - eps0):.16e}  (integrated with eps = {eps:.16e}, q = {q:.8f}, rho = {rho:.3e})")

    if Time_out is not None:
        test = Time_out

    return retval

# =========================
# Set parameters for minpack
# =========================
import time
from scipy.optimize import root
repeat_num_minpack  = 5000
# repeat_num_minpack  = 1

minpack_tol     = 1.0e-10
minpack_xtol    = 1.0e-10
minpack_maxfev  = 50000
minpack_factor  = 0.1
minpack_diag    = [1.0]*3 + [1.0]*3 + [1.0]*1
minpack_eps     = 0.
minpack_error   = 1.0e-7

np.set_printoptions(precision=16, suppress=True)

# =========================
# Convergence is judged by the physical residual at the target smoothing level only
# =========================
def physical_ok(lam, eps, lambda0 = 1.0):
    res = np.asarray(shooting(lam, eps, lambda0), dtype=float)
    return bool(np.all(np.isfinite(res)) and np.linalg.norm(res) < minpack_error
                and 0.0 < ta.state[6] <= 1.0)

In [6]:

def prepare_timing():
    set_timing_priority()
    if sys.platform != 'darwin':
        return
    started = time.perf_counter()
    deadline = started + 2.0
    cpu = ctypes.c_size_t()
    while True:
        lib.pthread_cpu_number_np(ctypes.byref(cpu))
        if cpu.value in fast_cpus and time.perf_counter()-started >= 0.05:
            return
        if time.perf_counter() > deadline:
            raise RuntimeError('No Super-core scheduling during timing preparation.')

In [7]:
# =========================
# Common random initial guess pool
# =========================
np.random.seed(322)
initial_guess_pool = []
for i in range(repeat_num_minpack):
    x0 = (np.random.random(7) - 0.5) * 2.0
    x0[-1] = np.random.random()          # λ_m ∈ (0,1)
    initial_guess_pool.append(x0.copy())
initial_guess_pool = np.asarray(initial_guess_pool)

# Augmented methods use the same base guesses plus the homotopy variable (q0 = 0, i.e. eps = eps0)
augmented_initial_guess_pool = np.zeros((repeat_num_minpack, 8))
augmented_initial_guess_pool[:, :7] = initial_guess_pool
augmented_initial_guess_pool[:, 7] = 0.0

print(f"Generated {repeat_num_minpack} shared initial guesses.")

Generated 5000 shared initial guesses.


In [8]:
# =========================
# Trial records
# =========================
# The trials of a method are independent. The saved outputs below were obtained by evaluating this
# same loop, one trial per call, in single-thread worker processes; the printed numbers are those of
# a serial run. The timing cells of each method are always measured serially.
def run_records(method, trial_fn):
    records = []
    for i in range(repeat_num_minpack):
        records.append(trial_fn(i))
        if (i + 1) % 500 == 0:
            print(f"{i+1} trials, {sum(r['accepted'] for r in records)} converged", flush=True)
    return records

def report_records(method, records):
    accepted = [r for r in records if r['accepted']]
    print(f"Solved: {len(accepted)} in {len(records)} ({len(accepted)/len(records):.4f})")
    if accepted:
        print(f"Average number of function calls for Minpack: {np.mean([r['nfev'] for r in records]):.0f}")
        print(f"Average number of function calls for convergent solution: {np.mean([r['nfev'] for r in accepted]):.0f}")
        best = max(accepted, key=lambda r: r['mass_kg'])
        print(f"Best final mass: {best['mass_kg']:.6f} kg (trial {best['trial']})")
    records_by_method[method] = records

## Method 1: normal shooting method (eps = 1)

In [9]:
# =========================
# Method 1: normal shooting at eps = 1
# =========================
epsf = 1.0e0
lambda0 = 1.0

def trial_single_eps1(i):
    # Random initial co-states
    x0 = initial_guess_pool[i].copy()

    # Use root's args parameter to pass fixed arguments
    r = root(
        shooting,
        x0,
        args = (epsf, lambda0, False, None),   # order: lam, eps, lambda0, out, Time_out
        method = 'hybr',
        tol = minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag,
            'eps': minpack_eps,
        }
    )

    ok = bool(np.all(np.isfinite(r.x)) and physical_ok(r.x, epsf, lambda0))
    mass = float(ta.state[6]*MUnit) if ok else None
    return dict(trial=int(i), accepted=ok, nfev=int(r.nfev), status=int(r.status),
                residual_norm=float(np.linalg.norm(r.fun)), eps=epsf, mass_kg=mass, x=r.x.tolist())

In [10]:
records = run_records('Single_eps1', trial_single_eps1)
report_records('Single_eps1', records)

500 trials, 174 converged
1000 trials, 343 converged
1500 trials, 523 converged
2000 trials, 713 converged
2500 trials, 911 converged
3000 trials, 1098 converged
3500 trials, 1268 converged
4000 trials, 1454 converged
4500 trials, 1640 converged
5000 trials, 1825 converged
Solved: 1825 in 5000 (0.3650)
Average number of function calls for Minpack: 83
Average number of function calls for convergent solution: 106
Best final mass: 2331.036218 kg (trial 1729)


In [11]:
# =========================
# Computational time: all / successful / failed attempts
# =========================
timing_ids = np.arange(500)
accepted = np.array([r['accepted'] for r in records_by_method['Single_eps1']], dtype=bool)
successful_ids = timing_ids[accepted[timing_ids]]
failed_ids = timing_ids[~accepted[timing_ids]]
cpu_samples.clear()

In [12]:
%%timeit -n 1 -r 7 -v timing_all prepare_timing()
for i in timing_ids:
    sample_cpu()
    trial_single_eps1(i)

11.2 s ± 29.4 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [13]:
%%timeit -n 1 -r 7 -v timing_successful prepare_timing()
for i in successful_ids:
    sample_cpu()
    trial_single_eps1(i)

4.13 s ± 26.9 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [14]:
%%timeit -n 1 -r 7 -v timing_failed prepare_timing()
for i in failed_ids:
    sample_cpu()
    trial_single_eps1(i)

7.16 s ± 34 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [15]:
all_ms = timing_all.average * 1000.0 / len(timing_ids)
p_success = accepted.mean()
results_by_method['Single_eps1'] = dict(
    method='Single shooting, epsilon=1', trials=len(accepted), accepted=int(accepted.sum()),
    first_ms=all_ms / p_success if p_success else float('inf'),
    successful_ms=timing_successful.average * 1000.0 / len(successful_ids) if len(successful_ids) else float('nan'),
    failed_ms=timing_failed.average * 1000.0 / len(failed_ids) if len(failed_ids) else float('nan'),
    all_ms=all_ms,
    nfev_successful=float(np.mean([r['nfev'] for r in records_by_method['Single_eps1'] if r['accepted']])) if accepted.any() else float('nan'),
    nfev_all=float(np.mean([r['nfev'] for r in records_by_method['Single_eps1']])),
    timing_trials=len(timing_ids), timing_successful_trials=len(successful_ids), timing_failed_trials=len(failed_ids),
    timeit_runs_seconds=timing_all.all_runs,
    timeit_successful_seconds=timing_successful.all_runs,
    timeit_failed_seconds=timing_failed.all_runs,
    cpu_samples=dict(Counter(cpu_samples)),
    fast_core_sample_pct=100.0 * sum(cpu in fast_cpus for cpu in cpu_samples) / len(cpu_samples) if cpu_samples else None)

## Method 1: normal shooting method (eps = 1e-5)

In [16]:
# =========================
# Method 1: normal shooting at eps = 1e-5
# =========================
epsf = 1.0e-5
lambda0 = 1.0

def trial_single_epsf(i):
    # Random initial co-states
    x0 = initial_guess_pool[i].copy()

    # Use root's args parameter to pass fixed arguments
    r = root(
        shooting,
        x0,
        args = (epsf, lambda0, False, None),   # order: lam, eps, lambda0, out, Time_out
        method = 'hybr',
        tol = minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag,
            'eps': minpack_eps,
        }
    )

    ok = bool(np.all(np.isfinite(r.x)) and physical_ok(r.x, epsf, lambda0))
    mass = float(ta.state[6]*MUnit) if ok else None
    return dict(trial=int(i), accepted=ok, nfev=int(r.nfev), status=int(r.status),
                residual_norm=float(np.linalg.norm(r.fun)), eps=epsf, mass_kg=mass, x=r.x.tolist())

In [17]:
records = run_records('Single_epsf', trial_single_epsf)
report_records('Single_epsf', records)

500 trials, 47 converged
1000 trials, 86 converged
1500 trials, 132 converged
2000 trials, 166 converged
2500 trials, 214 converged
3000 trials, 256 converged
3500 trials, 302 converged
4000 trials, 347 converged
4500 trials, 402 converged
5000 trials, 444 converged
Solved: 444 in 5000 (0.0888)
Average number of function calls for Minpack: 96
Average number of function calls for convergent solution: 171
Best final mass: 2718.288592 kg (trial 4716)


In [18]:
# =========================
# Computational time: all / successful / failed attempts
# =========================
timing_ids = np.arange(500)
accepted = np.array([r['accepted'] for r in records_by_method['Single_epsf']], dtype=bool)
successful_ids = timing_ids[accepted[timing_ids]]
failed_ids = timing_ids[~accepted[timing_ids]]
cpu_samples.clear()

In [19]:
%%timeit -n 1 -r 7 -v timing_all prepare_timing()
for i in timing_ids:
    sample_cpu()
    trial_single_epsf(i)

34 s ± 152 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [20]:
%%timeit -n 1 -r 7 -v timing_successful prepare_timing()
for i in successful_ids:
    sample_cpu()
    trial_single_epsf(i)

6.39 s ± 40.2 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [21]:
%%timeit -n 1 -r 7 -v timing_failed prepare_timing()
for i in failed_ids:
    sample_cpu()
    trial_single_epsf(i)

27.8 s ± 204 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [22]:
all_ms = timing_all.average * 1000.0 / len(timing_ids)
p_success = accepted.mean()
results_by_method['Single_epsf'] = dict(
    method='Single shooting, epsilon=1e-5', trials=len(accepted), accepted=int(accepted.sum()),
    first_ms=all_ms / p_success if p_success else float('inf'),
    successful_ms=timing_successful.average * 1000.0 / len(successful_ids) if len(successful_ids) else float('nan'),
    failed_ms=timing_failed.average * 1000.0 / len(failed_ids) if len(failed_ids) else float('nan'),
    all_ms=all_ms,
    nfev_successful=float(np.mean([r['nfev'] for r in records_by_method['Single_epsf'] if r['accepted']])) if accepted.any() else float('nan'),
    nfev_all=float(np.mean([r['nfev'] for r in records_by_method['Single_epsf']])),
    timing_trials=len(timing_ids), timing_successful_trials=len(successful_ids), timing_failed_trials=len(failed_ids),
    timeit_runs_seconds=timing_all.all_runs,
    timeit_successful_seconds=timing_successful.all_runs,
    timeit_failed_seconds=timing_failed.all_runs,
    cpu_samples=dict(Counter(cpu_samples)),
    fast_core_sample_pct=100.0 * sum(cpu in fast_cpus for cpu in cpu_samples) / len(cpu_samples) if cpu_samples else None)

## Method 2: consecutive iterations on the smoothing parameter

In [23]:
# =========================
# Method 2: consecutive iterations on the smoothing parameter
# =========================
# Continuation in eps: eps0 -> epsf
eps0 = 1.0e0
epsf = 1.0e-5
lambda0 = 1.0

# continuation/backtracking parameters (eps_next = eps_curr * d)
d0 = 0.1                 # initial step ratio (aggressive)
d_shrink = 0.8           # backtracking factor (used to INCREASE d so eps_next is closer to eps_curr)
d_recover = 0.8          # gentle recovery factor (after success, gradually DECREASE d back toward d0)
d_max = 0.999            # upper bound of d (avoid d==1 causing stagnation)
max_backtrack = 20       # max backtracking tries per step
max_total_steps = 300    # max accepted eps steps per continuation path (avoid infinite loops)

# wrap a single root() call to unify nfev and the success criterion
def solve_for_eps(x_init, eps_val):
    r = root(
        shooting,
        x_init,
        args=(eps_val, lambda0, False, None),  # (eps, lambda0, out, Time_out)
        method='hybr',
        tol=minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag,
            'eps': minpack_eps,
        }
    )

    ok = bool(np.all(np.isfinite(r.x)) and physical_ok(r.x, eps_val, lambda0))
    return r, ok

def trial_stepwise(i):
    # Random initial co-states
    x0 = initial_guess_pool[i].copy()

    # continuation initialization
    eps_curr = eps0
    d = d0
    x_guess = x0.copy()
    nfev = 0
    solved_this_outer = False

    # ============================================================
    # 0) Solve the easiest problem at eps0 ONCE (outside inner loop)
    # ============================================================
    r, ok0 = solve_for_eps(x_guess, eps_curr)
    nfev += r.nfev

    # ============================================================
    # 1) Continuation: advance eps until epsf (accepted eps only)
    # ============================================================
    if ok0:
        x_guess = r.x.copy()
        for step in range(max_total_steps):

            # Stop if target reached
            if eps_curr <= epsf:
                solved_this_outer = True
                break

            # Backtracking search for a feasible eps_next = eps_curr * d
            advanced = False
            for backtrack in range(max_backtrack + 1):
                eps_next = max(eps_curr * d, epsf)

                # Numerical safety: if eps_next does not change, force epsf
                if eps_next == eps_curr:
                    eps_next = epsf

                r, ok_next = solve_for_eps(x_guess, eps_next)
                nfev += r.nfev

                if ok_next:
                    # Accept eps_next
                    eps_curr = eps_next
                    x_guess = r.x.copy()
                    # gentle recovery of d (after success, gradually decrease d back toward d0)
                    d = max(d0, d * d_recover)
                    advanced = True
                    break
                else:
                    # Increase d to bring eps_next closer to eps_curr
                    d = min(d_max, d / d_shrink)

            if not advanced:
                # Too many backtracks and still cannot advance -> fail this random initial guess
                break

    solved = bool(solved_this_outer and physical_ok(x_guess, epsf, lambda0))
    mass = float(ta.state[6]*MUnit) if solved else None
    return dict(trial=int(i), accepted=solved, nfev=int(nfev), status=int(r.status),
                residual_norm=float(np.linalg.norm(r.fun)), eps=float(eps_curr), mass_kg=mass, x=x_guess.tolist())

In [24]:
records = run_records('Stepwise', trial_stepwise)
report_records('Stepwise', records)

500 trials, 174 converged
1000 trials, 343 converged
1500 trials, 523 converged
2000 trials, 713 converged
2500 trials, 911 converged
3000 trials, 1098 converged
3500 trials, 1268 converged
4000 trials, 1454 converged
4500 trials, 1640 converged
5000 trials, 1825 converged
Solved: 1825 in 5000 (0.3650)
Average number of function calls for Minpack: 129
Average number of function calls for convergent solution: 232
Best final mass: 2718.288592 kg (trial 309)


In [25]:
# =========================
# Computational time: all / successful / failed attempts
# =========================
timing_ids = np.arange(500)
accepted = np.array([r['accepted'] for r in records_by_method['Stepwise']], dtype=bool)
successful_ids = timing_ids[accepted[timing_ids]]
failed_ids = timing_ids[~accepted[timing_ids]]
cpu_samples.clear()

In [26]:
%%timeit -n 1 -r 7 -v timing_all prepare_timing()
for i in timing_ids:
    sample_cpu()
    trial_stepwise(i)

24.1 s ± 318 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [27]:
%%timeit -n 1 -r 7 -v timing_successful prepare_timing()
for i in successful_ids:
    sample_cpu()
    trial_stepwise(i)

16.8 s ± 193 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [28]:
%%timeit -n 1 -r 7 -v timing_failed prepare_timing()
for i in failed_ids:
    sample_cpu()
    trial_stepwise(i)

7.13 s ± 36.2 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [29]:
all_ms = timing_all.average * 1000.0 / len(timing_ids)
p_success = accepted.mean()
results_by_method['Stepwise'] = dict(
    method='Stepwise', trials=len(accepted), accepted=int(accepted.sum()),
    first_ms=all_ms / p_success if p_success else float('inf'),
    successful_ms=timing_successful.average * 1000.0 / len(successful_ids) if len(successful_ids) else float('nan'),
    failed_ms=timing_failed.average * 1000.0 / len(failed_ids) if len(failed_ids) else float('nan'),
    all_ms=all_ms,
    nfev_successful=float(np.mean([r['nfev'] for r in records_by_method['Stepwise'] if r['accepted']])) if accepted.any() else float('nan'),
    nfev_all=float(np.mean([r['nfev'] for r in records_by_method['Stepwise']])),
    timing_trials=len(timing_ids), timing_successful_trials=len(successful_ids), timing_failed_trials=len(failed_ids),
    timeit_runs_seconds=timing_all.all_runs,
    timeit_successful_seconds=timing_successful.all_runs,
    timeit_failed_seconds=timing_failed.all_runs,
    cpu_samples=dict(Counter(cpu_samples)),
    fast_core_sample_pct=100.0 * sum(cpu in fast_cpus for cpu in cpu_samples) / len(cpu_samples) if cpu_samples else None)

## Method 3: augmented shooting method

In [30]:
# =========================
# Method 3: augmented shooting, ungated (q = 1)
# =========================
eps0 = 1.0e0
epsf = 1.0e-5
lambda0 = 1.0

def trial_ungated(i):
    # Random initial co-states + one homotopy variable q
    x0 = augmented_initial_guess_pool[i].copy()

    r = root(
        shooting_augmented,
        x0,
        args = (eps0, epsf, lambda0, False, None),   # order: lam, eps0, epsf, lambda0, out, Time_out
        method = 'hybr',
        tol = minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag + [1.],
            'eps': minpack_eps,
        }
    )

    ok = bool(np.all(np.isfinite(r.x)) and physical_ok(r.x[:7], epsf, lambda0))
    mass = float(ta.state[6]*MUnit) if ok else None
    return dict(trial=int(i), accepted=ok, nfev=int(r.nfev), status=int(r.status),
                residual_norm=float(np.linalg.norm(r.fun)), eps=float(eps0 + r.x[7]*(epsf - eps0)),
                q=float(r.x[7]), mass_kg=mass, x=r.x.tolist())

In [31]:
records = run_records('Ungated', trial_ungated)
report_records('Ungated', records)

500 trials, 152 converged
1000 trials, 284 converged
1500 trials, 426 converged
2000 trials, 586 converged
2500 trials, 749 converged
3000 trials, 888 converged
3500 trials, 1042 converged
4000 trials, 1186 converged
4500 trials, 1336 converged
5000 trials, 1500 converged
Solved: 1500 in 5000 (0.3000)
Average number of function calls for Minpack: 96
Average number of function calls for convergent solution: 138
Best final mass: 2718.288592 kg (trial 232)


In [32]:
# =========================
# Computational time: all / successful / failed attempts
# =========================
timing_ids = np.arange(500)
accepted = np.array([r['accepted'] for r in records_by_method['Ungated']], dtype=bool)
successful_ids = timing_ids[accepted[timing_ids]]
failed_ids = timing_ids[~accepted[timing_ids]]
cpu_samples.clear()

In [33]:
%%timeit -n 1 -r 7 -v timing_all prepare_timing()
for i in timing_ids:
    sample_cpu()
    trial_ungated(i)

18.5 s ± 114 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [34]:
%%timeit -n 1 -r 7 -v timing_successful prepare_timing()
for i in successful_ids:
    sample_cpu()
    trial_ungated(i)

8.87 s ± 49.8 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [35]:
%%timeit -n 1 -r 7 -v timing_failed prepare_timing()
for i in failed_ids:
    sample_cpu()
    trial_ungated(i)

9.6 s ± 46.9 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [36]:
all_ms = timing_all.average * 1000.0 / len(timing_ids)
p_success = accepted.mean()
results_by_method['Ungated'] = dict(
    method='Ungated', trials=len(accepted), accepted=int(accepted.sum()),
    first_ms=all_ms / p_success if p_success else float('inf'),
    successful_ms=timing_successful.average * 1000.0 / len(successful_ids) if len(successful_ids) else float('nan'),
    failed_ms=timing_failed.average * 1000.0 / len(failed_ids) if len(failed_ids) else float('nan'),
    all_ms=all_ms,
    nfev_successful=float(np.mean([r['nfev'] for r in records_by_method['Ungated'] if r['accepted']])) if accepted.any() else float('nan'),
    nfev_all=float(np.mean([r['nfev'] for r in records_by_method['Ungated']])),
    timing_trials=len(timing_ids), timing_successful_trials=len(successful_ids), timing_failed_trials=len(failed_ids),
    timeit_runs_seconds=timing_all.all_runs,
    timeit_successful_seconds=timing_successful.all_runs,
    timeit_failed_seconds=timing_failed.all_runs,
    cpu_samples=dict(Counter(cpu_samples)),
    fast_core_sample_pct=100.0 * sum(cpu in fast_cpus for cpu in cpu_samples) / len(cpu_samples) if cpu_samples else None)

## Method 3: augmented shooting method with exponential gate (p = 1)

In [37]:
# =========================
# Method 3: augmented shooting with the residual gate (exponential, p = 1)
# =========================
eps0 = 1.0e0
epsf = 1.0e-5

# parameters for the augmented residual
gate_option = 'gaussian'
gate_p = 1.0
rho0 = 0.316796

# [other parameters]:
fd_eps = 0.
delta_rho = 1.0e-8

# [general settings]:
lambda0 = 1.0

def trial_gate_exp_p1(i):
    # Random initial co-states + one homotopy variable q
    x0 = augmented_initial_guess_pool[i].copy()

    r = root(
        shooting_augmented_gate,
        x0,
        # pass eps0, epsf, and gate parameters
        args = (eps0, epsf, rho0, delta_rho, gate_p, gate_option, lambda0, False, None),
        method = 'hybr',
        tol = minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag + [1.],
            # 'eps': minpack_eps,
            'eps': fd_eps,
        }
    )

    ok = bool(np.all(np.isfinite(r.x)) and physical_ok(r.x[:7], epsf, lambda0))
    mass = float(ta.state[6]*MUnit) if ok else None
    return dict(trial=int(i), accepted=ok, nfev=int(r.nfev), status=int(r.status),
                residual_norm=float(np.linalg.norm(r.fun)), eps=float(eps0 + r.x[7]*(epsf - eps0)),
                q=float(r.x[7]), mass_kg=mass, x=r.x.tolist())

In [38]:
records = run_records('Gated_Exp_p1', trial_gate_exp_p1)
report_records('Gated_Exp_p1', records)

500 trials, 190 converged
1000 trials, 371 converged
1500 trials, 539 converged
2000 trials, 743 converged
2500 trials, 940 converged
3000 trials, 1139 converged
3500 trials, 1322 converged
4000 trials, 1490 converged
4500 trials, 1671 converged
5000 trials, 1863 converged
Solved: 1863 in 5000 (0.3726)
Average number of function calls for Minpack: 113
Average number of function calls for convergent solution: 178
Best final mass: 2718.288638 kg (trial 1408)


In [39]:
# =========================
# Computational time: all / successful / failed attempts
# =========================
timing_ids = np.arange(500)
accepted = np.array([r['accepted'] for r in records_by_method['Gated_Exp_p1']], dtype=bool)
successful_ids = timing_ids[accepted[timing_ids]]
failed_ids = timing_ids[~accepted[timing_ids]]
cpu_samples.clear()

In [40]:
%%timeit -n 1 -r 7 -v timing_all prepare_timing()
for i in timing_ids:
    sample_cpu()
    trial_gate_exp_p1(i)

20.6 s ± 95 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [41]:
%%timeit -n 1 -r 7 -v timing_successful prepare_timing()
for i in successful_ids:
    sample_cpu()
    trial_gate_exp_p1(i)

13.1 s ± 82.5 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [42]:
%%timeit -n 1 -r 7 -v timing_failed prepare_timing()
for i in failed_ids:
    sample_cpu()
    trial_gate_exp_p1(i)

7.6 s ± 90.4 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [43]:
all_ms = timing_all.average * 1000.0 / len(timing_ids)
p_success = accepted.mean()
results_by_method['Gated_Exp_p1'] = dict(
    method='Augmented, W_exp, p=1', trials=len(accepted), accepted=int(accepted.sum()),
    first_ms=all_ms / p_success if p_success else float('inf'),
    successful_ms=timing_successful.average * 1000.0 / len(successful_ids) if len(successful_ids) else float('nan'),
    failed_ms=timing_failed.average * 1000.0 / len(failed_ids) if len(failed_ids) else float('nan'),
    all_ms=all_ms,
    nfev_successful=float(np.mean([r['nfev'] for r in records_by_method['Gated_Exp_p1'] if r['accepted']])) if accepted.any() else float('nan'),
    nfev_all=float(np.mean([r['nfev'] for r in records_by_method['Gated_Exp_p1']])),
    timing_trials=len(timing_ids), timing_successful_trials=len(successful_ids), timing_failed_trials=len(failed_ids),
    timeit_runs_seconds=timing_all.all_runs,
    timeit_successful_seconds=timing_successful.all_runs,
    timeit_failed_seconds=timing_failed.all_runs,
    cpu_samples=dict(Counter(cpu_samples)),
    fast_core_sample_pct=100.0 * sum(cpu in fast_cpus for cpu in cpu_samples) / len(cpu_samples) if cpu_samples else None)

## Method 3: augmented shooting method with exponential gate (p = 2)

In [44]:
# =========================
# Method 3: augmented shooting with the residual gate (exponential, p = 2)
# =========================
eps0 = 1.0e0
epsf = 1.0e-5

# parameters for the augmented residual
gate_option = 'gaussian'
gate_p = 2.0
rho0 = 0.737159

# [other parameters]:
fd_eps = 0.
delta_rho = 1.0e-8

# [general settings]:
lambda0 = 1.0

def trial_gate_exp_p2(i):
    # Random initial co-states + one homotopy variable q
    x0 = augmented_initial_guess_pool[i].copy()

    r = root(
        shooting_augmented_gate,
        x0,
        # pass eps0, epsf, and gate parameters
        args = (eps0, epsf, rho0, delta_rho, gate_p, gate_option, lambda0, False, None),
        method = 'hybr',
        tol = minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag + [1.],
            # 'eps': minpack_eps,
            'eps': fd_eps,
        }
    )

    ok = bool(np.all(np.isfinite(r.x)) and physical_ok(r.x[:7], epsf, lambda0))
    mass = float(ta.state[6]*MUnit) if ok else None
    return dict(trial=int(i), accepted=ok, nfev=int(r.nfev), status=int(r.status),
                residual_norm=float(np.linalg.norm(r.fun)), eps=float(eps0 + r.x[7]*(epsf - eps0)),
                q=float(r.x[7]), mass_kg=mass, x=r.x.tolist())

In [45]:
records = run_records('Gated_Exp_p2', trial_gate_exp_p2)
report_records('Gated_Exp_p2', records)

500 trials, 184 converged
1000 trials, 361 converged
1500 trials, 522 converged
2000 trials, 724 converged
2500 trials, 906 converged
3000 trials, 1105 converged
3500 trials, 1285 converged
4000 trials, 1455 converged
4500 trials, 1634 converged
5000 trials, 1825 converged
Solved: 1825 in 5000 (0.3650)
Average number of function calls for Minpack: 98
Average number of function calls for convergent solution: 136
Best final mass: 2718.288592 kg (trial 4778)


In [46]:
# =========================
# Computational time: all / successful / failed attempts
# =========================
timing_ids = np.arange(500)
accepted = np.array([r['accepted'] for r in records_by_method['Gated_Exp_p2']], dtype=bool)
successful_ids = timing_ids[accepted[timing_ids]]
failed_ids = timing_ids[~accepted[timing_ids]]
cpu_samples.clear()

In [47]:
%%timeit -n 1 -r 7 -v timing_all prepare_timing()
for i in timing_ids:
    sample_cpu()
    trial_gate_exp_p2(i)

17.1 s ± 97.1 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [48]:
%%timeit -n 1 -r 7 -v timing_successful prepare_timing()
for i in successful_ids:
    sample_cpu()
    trial_gate_exp_p2(i)

9.19 s ± 82 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [49]:
%%timeit -n 1 -r 7 -v timing_failed prepare_timing()
for i in failed_ids:
    sample_cpu()
    trial_gate_exp_p2(i)

7.82 s ± 9.09 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [50]:
all_ms = timing_all.average * 1000.0 / len(timing_ids)
p_success = accepted.mean()
results_by_method['Gated_Exp_p2'] = dict(
    method='Augmented, W_exp, p=2', trials=len(accepted), accepted=int(accepted.sum()),
    first_ms=all_ms / p_success if p_success else float('inf'),
    successful_ms=timing_successful.average * 1000.0 / len(successful_ids) if len(successful_ids) else float('nan'),
    failed_ms=timing_failed.average * 1000.0 / len(failed_ids) if len(failed_ids) else float('nan'),
    all_ms=all_ms,
    nfev_successful=float(np.mean([r['nfev'] for r in records_by_method['Gated_Exp_p2'] if r['accepted']])) if accepted.any() else float('nan'),
    nfev_all=float(np.mean([r['nfev'] for r in records_by_method['Gated_Exp_p2']])),
    timing_trials=len(timing_ids), timing_successful_trials=len(successful_ids), timing_failed_trials=len(failed_ids),
    timeit_runs_seconds=timing_all.all_runs,
    timeit_successful_seconds=timing_successful.all_runs,
    timeit_failed_seconds=timing_failed.all_runs,
    cpu_samples=dict(Counter(cpu_samples)),
    fast_core_sample_pct=100.0 * sum(cpu in fast_cpus for cpu in cpu_samples) / len(cpu_samples) if cpu_samples else None)

## Method 3: augmented shooting method with exponential gate (p = 4)

In [51]:
# =========================
# Method 3: augmented shooting with the residual gate (exponential, p = 4)
# =========================
eps0 = 1.0e0
epsf = 1.0e-5

# parameters for the augmented residual
gate_option = 'gaussian'
gate_p = 4.0
rho0 = 0.293384

# [other parameters]:
fd_eps = 0.
delta_rho = 1.0e-8

# [general settings]:
lambda0 = 1.0

def trial_gate_exp_p4(i):
    # Random initial co-states + one homotopy variable q
    x0 = augmented_initial_guess_pool[i].copy()

    r = root(
        shooting_augmented_gate,
        x0,
        # pass eps0, epsf, and gate parameters
        args = (eps0, epsf, rho0, delta_rho, gate_p, gate_option, lambda0, False, None),
        method = 'hybr',
        tol = minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag + [1.],
            # 'eps': minpack_eps,
            'eps': fd_eps,
        }
    )

    ok = bool(np.all(np.isfinite(r.x)) and physical_ok(r.x[:7], epsf, lambda0))
    mass = float(ta.state[6]*MUnit) if ok else None
    return dict(trial=int(i), accepted=ok, nfev=int(r.nfev), status=int(r.status),
                residual_norm=float(np.linalg.norm(r.fun)), eps=float(eps0 + r.x[7]*(epsf - eps0)),
                q=float(r.x[7]), mass_kg=mass, x=r.x.tolist())

In [52]:
records = run_records('Gated_Exp_p4', trial_gate_exp_p4)
report_records('Gated_Exp_p4', records)

500 trials, 184 converged
1000 trials, 360 converged
1500 trials, 528 converged
2000 trials, 733 converged
2500 trials, 922 converged
3000 trials, 1119 converged
3500 trials, 1295 converged
4000 trials, 1464 converged
4500 trials, 1646 converged
5000 trials, 1830 converged
Solved: 1830 in 5000 (0.3660)
Average number of function calls for Minpack: 101
Average number of function calls for convergent solution: 144
Best final mass: 2718.288592 kg (trial 262)


In [53]:
# =========================
# Computational time: all / successful / failed attempts
# =========================
timing_ids = np.arange(500)
accepted = np.array([r['accepted'] for r in records_by_method['Gated_Exp_p4']], dtype=bool)
successful_ids = timing_ids[accepted[timing_ids]]
failed_ids = timing_ids[~accepted[timing_ids]]
cpu_samples.clear()

In [54]:
%%timeit -n 1 -r 7 -v timing_all prepare_timing()
for i in timing_ids:
    sample_cpu()
    trial_gate_exp_p4(i)

17.3 s ± 81.3 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [55]:
%%timeit -n 1 -r 7 -v timing_successful prepare_timing()
for i in successful_ids:
    sample_cpu()
    trial_gate_exp_p4(i)

9.5 s ± 16 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [56]:
%%timeit -n 1 -r 7 -v timing_failed prepare_timing()
for i in failed_ids:
    sample_cpu()
    trial_gate_exp_p4(i)

7.79 s ± 57.2 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [57]:
all_ms = timing_all.average * 1000.0 / len(timing_ids)
p_success = accepted.mean()
results_by_method['Gated_Exp_p4'] = dict(
    method='Augmented, W_exp, p=4', trials=len(accepted), accepted=int(accepted.sum()),
    first_ms=all_ms / p_success if p_success else float('inf'),
    successful_ms=timing_successful.average * 1000.0 / len(successful_ids) if len(successful_ids) else float('nan'),
    failed_ms=timing_failed.average * 1000.0 / len(failed_ids) if len(failed_ids) else float('nan'),
    all_ms=all_ms,
    nfev_successful=float(np.mean([r['nfev'] for r in records_by_method['Gated_Exp_p4'] if r['accepted']])) if accepted.any() else float('nan'),
    nfev_all=float(np.mean([r['nfev'] for r in records_by_method['Gated_Exp_p4']])),
    timing_trials=len(timing_ids), timing_successful_trials=len(successful_ids), timing_failed_trials=len(failed_ids),
    timeit_runs_seconds=timing_all.all_runs,
    timeit_successful_seconds=timing_successful.all_runs,
    timeit_failed_seconds=timing_failed.all_runs,
    cpu_samples=dict(Counter(cpu_samples)),
    fast_core_sample_pct=100.0 * sum(cpu in fast_cpus for cpu in cpu_samples) / len(cpu_samples) if cpu_samples else None)

## Method 3: augmented shooting method with rational gate (p = 1)

In [58]:
# =========================
# Method 3: augmented shooting with the residual gate (rational, p = 1)
# =========================
eps0 = 1.0e0
epsf = 1.0e-5

# parameters for the augmented residual
gate_option = 'rational'
gate_p = 1.0
rho0 = 0.233027

# [other parameters]:
fd_eps = 0.
delta_rho = 1.0e-8

# [general settings]:
lambda0 = 1.0

def trial_gate_rational_p1(i):
    # Random initial co-states + one homotopy variable q
    x0 = augmented_initial_guess_pool[i].copy()

    r = root(
        shooting_augmented_gate,
        x0,
        # pass eps0, epsf, and gate parameters
        args = (eps0, epsf, rho0, delta_rho, gate_p, gate_option, lambda0, False, None),
        method = 'hybr',
        tol = minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag + [1.],
            # 'eps': minpack_eps,
            'eps': fd_eps,
        }
    )

    ok = bool(np.all(np.isfinite(r.x)) and physical_ok(r.x[:7], epsf, lambda0))
    mass = float(ta.state[6]*MUnit) if ok else None
    return dict(trial=int(i), accepted=ok, nfev=int(r.nfev), status=int(r.status),
                residual_norm=float(np.linalg.norm(r.fun)), eps=float(eps0 + r.x[7]*(epsf - eps0)),
                q=float(r.x[7]), mass_kg=mass, x=r.x.tolist())

In [59]:
records = run_records('Gated_Rational_p1', trial_gate_rational_p1)
report_records('Gated_Rational_p1', records)

500 trials, 179 converged
1000 trials, 366 converged
1500 trials, 534 converged
2000 trials, 727 converged
2500 trials, 931 converged
3000 trials, 1111 converged
3500 trials, 1280 converged
4000 trials, 1448 converged
4500 trials, 1626 converged
5000 trials, 1820 converged
Solved: 1820 in 5000 (0.3640)
Average number of function calls for Minpack: 115
Average number of function calls for convergent solution: 186
Best final mass: 2718.288654 kg (trial 3977)


In [60]:
# =========================
# Computational time: all / successful / failed attempts
# =========================
timing_ids = np.arange(500)
accepted = np.array([r['accepted'] for r in records_by_method['Gated_Rational_p1']], dtype=bool)
successful_ids = timing_ids[accepted[timing_ids]]
failed_ids = timing_ids[~accepted[timing_ids]]
cpu_samples.clear()

In [61]:
%%timeit -n 1 -r 7 -v timing_all prepare_timing()
for i in timing_ids:
    sample_cpu()
    trial_gate_rational_p1(i)

20.6 s ± 73.6 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [62]:
%%timeit -n 1 -r 7 -v timing_successful prepare_timing()
for i in successful_ids:
    sample_cpu()
    trial_gate_rational_p1(i)

12.9 s ± 41 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [63]:
%%timeit -n 1 -r 7 -v timing_failed prepare_timing()
for i in failed_ids:
    sample_cpu()
    trial_gate_rational_p1(i)

7.74 s ± 26.2 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [64]:
all_ms = timing_all.average * 1000.0 / len(timing_ids)
p_success = accepted.mean()
results_by_method['Gated_Rational_p1'] = dict(
    method='Augmented, W_rat, p=1', trials=len(accepted), accepted=int(accepted.sum()),
    first_ms=all_ms / p_success if p_success else float('inf'),
    successful_ms=timing_successful.average * 1000.0 / len(successful_ids) if len(successful_ids) else float('nan'),
    failed_ms=timing_failed.average * 1000.0 / len(failed_ids) if len(failed_ids) else float('nan'),
    all_ms=all_ms,
    nfev_successful=float(np.mean([r['nfev'] for r in records_by_method['Gated_Rational_p1'] if r['accepted']])) if accepted.any() else float('nan'),
    nfev_all=float(np.mean([r['nfev'] for r in records_by_method['Gated_Rational_p1']])),
    timing_trials=len(timing_ids), timing_successful_trials=len(successful_ids), timing_failed_trials=len(failed_ids),
    timeit_runs_seconds=timing_all.all_runs,
    timeit_successful_seconds=timing_successful.all_runs,
    timeit_failed_seconds=timing_failed.all_runs,
    cpu_samples=dict(Counter(cpu_samples)),
    fast_core_sample_pct=100.0 * sum(cpu in fast_cpus for cpu in cpu_samples) / len(cpu_samples) if cpu_samples else None)

## Method 3: augmented shooting method with rational gate (p = 2)

In [65]:
# =========================
# Method 3: augmented shooting with the residual gate (rational, p = 2)
# =========================
eps0 = 1.0e0
epsf = 1.0e-5

# parameters for the augmented residual
gate_option = 'rational'
gate_p = 2.0
rho0 = 0.185086

# [other parameters]:
fd_eps = 0.
delta_rho = 1.0e-8

# [general settings]:
lambda0 = 1.0

def trial_gate_rational_p2(i):
    # Random initial co-states + one homotopy variable q
    x0 = augmented_initial_guess_pool[i].copy()

    r = root(
        shooting_augmented_gate,
        x0,
        # pass eps0, epsf, and gate parameters
        args = (eps0, epsf, rho0, delta_rho, gate_p, gate_option, lambda0, False, None),
        method = 'hybr',
        tol = minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag + [1.],
            # 'eps': minpack_eps,
            'eps': fd_eps,
        }
    )

    ok = bool(np.all(np.isfinite(r.x)) and physical_ok(r.x[:7], epsf, lambda0))
    mass = float(ta.state[6]*MUnit) if ok else None
    return dict(trial=int(i), accepted=ok, nfev=int(r.nfev), status=int(r.status),
                residual_norm=float(np.linalg.norm(r.fun)), eps=float(eps0 + r.x[7]*(epsf - eps0)),
                q=float(r.x[7]), mass_kg=mass, x=r.x.tolist())

In [66]:
records = run_records('Gated_Rational_p2', trial_gate_rational_p2)
report_records('Gated_Rational_p2', records)

500 trials, 191 converged
1000 trials, 363 converged
1500 trials, 534 converged
2000 trials, 734 converged
2500 trials, 922 converged
3000 trials, 1098 converged
3500 trials, 1276 converged
4000 trials, 1456 converged
4500 trials, 1629 converged
5000 trials, 1824 converged
Solved: 1824 in 5000 (0.3648)
Average number of function calls for Minpack: 103
Average number of function calls for convergent solution: 152
Best final mass: 2718.288592 kg (trial 364)


In [67]:
# =========================
# Computational time: all / successful / failed attempts
# =========================
timing_ids = np.arange(500)
accepted = np.array([r['accepted'] for r in records_by_method['Gated_Rational_p2']], dtype=bool)
successful_ids = timing_ids[accepted[timing_ids]]
failed_ids = timing_ids[~accepted[timing_ids]]
cpu_samples.clear()

In [68]:
%%timeit -n 1 -r 7 -v timing_all prepare_timing()
for i in timing_ids:
    sample_cpu()
    trial_gate_rational_p2(i)

18 s ± 311 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [69]:
%%timeit -n 1 -r 7 -v timing_successful prepare_timing()
for i in successful_ids:
    sample_cpu()
    trial_gate_rational_p2(i)

10.6 s ± 204 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [70]:
%%timeit -n 1 -r 7 -v timing_failed prepare_timing()
for i in failed_ids:
    sample_cpu()
    trial_gate_rational_p2(i)

7.76 s ± 138 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [71]:
all_ms = timing_all.average * 1000.0 / len(timing_ids)
p_success = accepted.mean()
results_by_method['Gated_Rational_p2'] = dict(
    method='Augmented, W_rat, p=2', trials=len(accepted), accepted=int(accepted.sum()),
    first_ms=all_ms / p_success if p_success else float('inf'),
    successful_ms=timing_successful.average * 1000.0 / len(successful_ids) if len(successful_ids) else float('nan'),
    failed_ms=timing_failed.average * 1000.0 / len(failed_ids) if len(failed_ids) else float('nan'),
    all_ms=all_ms,
    nfev_successful=float(np.mean([r['nfev'] for r in records_by_method['Gated_Rational_p2'] if r['accepted']])) if accepted.any() else float('nan'),
    nfev_all=float(np.mean([r['nfev'] for r in records_by_method['Gated_Rational_p2']])),
    timing_trials=len(timing_ids), timing_successful_trials=len(successful_ids), timing_failed_trials=len(failed_ids),
    timeit_runs_seconds=timing_all.all_runs,
    timeit_successful_seconds=timing_successful.all_runs,
    timeit_failed_seconds=timing_failed.all_runs,
    cpu_samples=dict(Counter(cpu_samples)),
    fast_core_sample_pct=100.0 * sum(cpu in fast_cpus for cpu in cpu_samples) / len(cpu_samples) if cpu_samples else None)

## Method 3: augmented shooting method with rational gate (p = 4)

In [72]:
# =========================
# Method 3: augmented shooting with the residual gate (rational, p = 4)
# =========================
eps0 = 1.0e0
epsf = 1.0e-5

# parameters for the augmented residual
gate_option = 'rational'
gate_p = 4.0
rho0 = 0.171408

# [other parameters]:
fd_eps = 0.
delta_rho = 1.0e-8

# [general settings]:
lambda0 = 1.0

def trial_gate_rational_p4(i):
    # Random initial co-states + one homotopy variable q
    x0 = augmented_initial_guess_pool[i].copy()

    r = root(
        shooting_augmented_gate,
        x0,
        # pass eps0, epsf, and gate parameters
        args = (eps0, epsf, rho0, delta_rho, gate_p, gate_option, lambda0, False, None),
        method = 'hybr',
        tol = minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag + [1.],
            # 'eps': minpack_eps,
            'eps': fd_eps,
        }
    )

    ok = bool(np.all(np.isfinite(r.x)) and physical_ok(r.x[:7], epsf, lambda0))
    mass = float(ta.state[6]*MUnit) if ok else None
    return dict(trial=int(i), accepted=ok, nfev=int(r.nfev), status=int(r.status),
                residual_norm=float(np.linalg.norm(r.fun)), eps=float(eps0 + r.x[7]*(epsf - eps0)),
                q=float(r.x[7]), mass_kg=mass, x=r.x.tolist())

In [73]:
records = run_records('Gated_Rational_p4', trial_gate_rational_p4)
report_records('Gated_Rational_p4', records)

500 trials, 186 converged
1000 trials, 370 converged
1500 trials, 541 converged
2000 trials, 751 converged
2500 trials, 946 converged
3000 trials, 1139 converged
3500 trials, 1318 converged
4000 trials, 1493 converged
4500 trials, 1674 converged
5000 trials, 1863 converged
Solved: 1863 in 5000 (0.3726)
Average number of function calls for Minpack: 102
Average number of function calls for convergent solution: 148
Best final mass: 2718.288592 kg (trial 1944)


In [74]:
# =========================
# Computational time: all / successful / failed attempts
# =========================
timing_ids = np.arange(500)
accepted = np.array([r['accepted'] for r in records_by_method['Gated_Rational_p4']], dtype=bool)
successful_ids = timing_ids[accepted[timing_ids]]
failed_ids = timing_ids[~accepted[timing_ids]]
cpu_samples.clear()

In [75]:
%%timeit -n 1 -r 7 -v timing_all prepare_timing()
for i in timing_ids:
    sample_cpu()
    trial_gate_rational_p4(i)

17.3 s ± 54.1 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [76]:
%%timeit -n 1 -r 7 -v timing_successful prepare_timing()
for i in successful_ids:
    sample_cpu()
    trial_gate_rational_p4(i)

9.68 s ± 231 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [77]:
%%timeit -n 1 -r 7 -v timing_failed prepare_timing()
for i in failed_ids:
    sample_cpu()
    trial_gate_rational_p4(i)

7.77 s ± 94.2 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [78]:
all_ms = timing_all.average * 1000.0 / len(timing_ids)
p_success = accepted.mean()
results_by_method['Gated_Rational_p4'] = dict(
    method='Augmented, W_rat, p=4', trials=len(accepted), accepted=int(accepted.sum()),
    first_ms=all_ms / p_success if p_success else float('inf'),
    successful_ms=timing_successful.average * 1000.0 / len(successful_ids) if len(successful_ids) else float('nan'),
    failed_ms=timing_failed.average * 1000.0 / len(failed_ids) if len(failed_ids) else float('nan'),
    all_ms=all_ms,
    nfev_successful=float(np.mean([r['nfev'] for r in records_by_method['Gated_Rational_p4'] if r['accepted']])) if accepted.any() else float('nan'),
    nfev_all=float(np.mean([r['nfev'] for r in records_by_method['Gated_Rational_p4']])),
    timing_trials=len(timing_ids), timing_successful_trials=len(successful_ids), timing_failed_trials=len(failed_ids),
    timeit_runs_seconds=timing_all.all_runs,
    timeit_successful_seconds=timing_successful.all_runs,
    timeit_failed_seconds=timing_failed.all_runs,
    cpu_samples=dict(Counter(cpu_samples)),
    fast_core_sample_pct=100.0 * sum(cpu in fast_cpus for cpu in cpu_samples) / len(cpu_samples) if cpu_samples else None)

## Summary

In [79]:

headings = ['Method', 'Success / 5000', 'Convergence [%]', 'Expected first [ms]',
            'Successful mean [ms]', 'Failed mean [ms]', 'All-trial mean [ms]', 'Nfev (all)', 'Nfev (converged)']
table = ['| '+' | '.join(headings)+' |', '| '+' | '.join(['---']*len(headings))+' |']
for value in results_by_method.values():
    values = [value['method'], value['accepted'], f"{100.*value['accepted']/value['trials']:.2f}"]
    values += [f"{value[k]:.3f}" for k in ('first_ms','successful_ms','failed_ms','all_ms')]
    values += [f"{value['nfev_all']:.1f}", f"{value['nfev_successful']:.1f}"]
    table.append('| '+' | '.join(map(str,values))+' |')
display(Markdown('\n'.join(table)))

| Method | Success / 5000 | Convergence [%] | Expected first [ms] | Successful mean [ms] | Failed mean [ms] | All-trial mean [ms] | Nfev (all) | Nfev (converged) |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| Single shooting, epsilon=1 | 1825 | 36.50 | 61.626 | 23.749 | 21.950 | 22.493 | 83.3 | 105.9 |
| Single shooting, epsilon=1e-5 | 444 | 8.88 | 765.930 | 135.913 | 61.290 | 68.015 | 96.0 | 170.9 |
| Stepwise | 1825 | 36.50 | 132.017 | 96.460 | 21.885 | 48.186 | 129.3 | 231.9 |
| Ungated | 1500 | 30.00 | 123.530 | 58.363 | 27.584 | 37.059 | 96.3 | 138.3 |
| Augmented, W_exp, p=1 | 1863 | 37.26 | 110.762 | 68.725 | 24.519 | 41.270 | 113.5 | 178.5 |
| Augmented, W_exp, p=2 | 1825 | 36.50 | 93.639 | 49.954 | 24.736 | 34.178 | 98.0 | 136.4 |
| Augmented, W_exp, p=4 | 1830 | 36.60 | 94.506 | 51.651 | 24.658 | 34.589 | 100.6 | 143.7 |
| Augmented, W_rat, p=1 | 1820 | 36.40 | 113.024 | 71.806 | 24.118 | 41.141 | 115.3 | 186.4 |
| Augmented, W_rat, p=2 | 1824 | 36.48 | 98.680 | 55.609 | 25.114 | 35.999 | 103.0 | 151.5 |
| Augmented, W_rat, p=4 | 1863 | 37.26 | 93.004 | 52.065 | 24.753 | 34.653 | 102.3 | 148.0 |